# Project 4B: 3D LiDAR-Inertial SLAM

![Map of Bruin Plaza at UCLA](media/bruin_plaza_pcd.png)
Final stitched global point-cloud map of Bruin Plaza at UCLA using ISAM2.

### What You Are Building

In Project 4A, you explored a variant of ICP called Generalized-ICP, enabling you to register scans with more flexibility and robustness than the traditional point-to-point algorithm. These capabilities are crucial when working with 3D data, such as on a drone. In this project, we build upon scan-to-scan registration for development of a LiDAR-inertial SLAM pipeline, using Open3D for fast and optimized GICP registration and GTSAM to manage the nonlinear optimization.

You are completing a LiDAR-inertial odometry / SLAM pipeline. At a high level, the system:

- reads IMU and LiDAR data from a ROS bag,
- estimates motion between LiDAR scans,
- uses IMU preintegration to summarize high-rate IMU data between keyframes,
- builds a factor graph in GTSAM, and
- compares a batch backend against an incremental ISAM2 backend.

Most of your work is in a small set of TODO blocks marked by `STUDENT TODO START` / `STUDENT TODO END`.

Focus on these files:

- [src/lio_common.py](src/lio_common.py): IMU buffering, preintegration, and IMU-related graph factors
- [src/lio_batch.py](src/lio_batch.py): batch backend initialization and optimization updates
- [src/lio_isam2.py](src/lio_isam2.py): incremental ISAM2 backend updates
- [src/lio_results.py](src/lio_results.py): stitched global map creation

You do **not** need to understand every file in `src/` before starting. The notebook introduces the required pieces in a reasonable order.

### Data

The provided dataset and a reference trajectory are available for download in [OneDrive](https://gtvault-my.sharepoint.com/:f:/g/personal/mwoodward8_gatech_edu/IgC-Nm7BEu1CT4NIw_A5cUdMAWg4LyYc8e2W5Hx-fpRjrFY?e=JiftYr). Please download the folders within and place them directly into a `data/` folder.

**The LiDAR & IMU data are provided by the authors of [DLIO](https://arxiv.org/abs/2203.03749) and contain aerial data of regions of the UCLA campus!**

The reference trajectory was computed with [rko_lio](https://arxiv.org/abs/2509.06593) ([GitHub](https://github.com/PRBonn/rko_lio?tab=readme-ov-file)). This is a recent and highly performant LIO system with a very easy to use Python package. If LIO interests you, I would encourage you to read their paper, try our dataset with their Python package, and compare your implementation here to their SOTA system!

### What To Turn In

Code submission (4B):

- [src/lio_common.py](src/lio_common.py)
- [src/lio_results.py](src/lio_results.py)

Reflection submission (4C):

- reflection slides in PDF format

**Project 4B is due on Tuesday, April 28th at 11:59 PM EST.**

### Author

Created by Matthew Woodward. Please reach out on Piazza if you need any assistance or find any bugs.

## Setup and Environment

First, create and activate the Conda environment:
```bash
conda env create -f environment.yml
conda activate amr_p4b
```
Then run the cell below.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from copy import deepcopy
from src.lio_common import LioSlamConfig
import matplotlib.pyplot as plt
import numpy as np
from statistics import mean, median

cfg = LioSlamConfig()
cfg.use_preintegration = True
cfg.use_ground_truth_slice_initialization = True

def run_backend(backend_cls, config, label):
    try:
        slam = backend_cls(config=config)
        result = slam.run()
        print(f'{label} run complete: {len(result.keyframes)} keyframes')
        return slam, result
    except NotImplementedError as exc:
        print(f'{label} hit a starter-code TODO: {exc}')
    except KeyboardInterrupt:
        print(f'{label} run interrupted')
    except Exception as exc:
        print(f'{label} run failed: {type(exc).__name__}: {exc}')
    return None, None

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Implement Common Functionality

The local test suite is intentionally small. It is meant to help you verify the core assignment pieces before you run the full pipeline.
These tests do not fully validate SLAM quality. Passing them is necessary, but not sufficient, for the full assignment.

Tests will require you to implement the following functions:
- [src/lio_common.py](src/lio_common.py):
    - (5 pts) `handle_imu_measurement()`
    - (5 pts) `add_imu_factor()`
    - (5 pts) `add_bias_evolution_factor()`
    - (5 pts) `predict_navstate_from_preintegration()`
- [src/lio_results.py](src/lio_results.py):
    - (5 pts) `build_global_map()`

Complete these functions before moving on and run the cell for unit test verification. These tests determine your **Code** score on Gradescope for this assignment. The rest of the points will come from reflection on your full pipeline.

In [ ]:
# Run the local student unit tests
import subprocess

test_cmd = [
    sys.executable,
    '-m',
    'unittest',
    'discover',
    '-s',
    'tests',
    '-p',
    'test_student_suite.py',
    '-v',
]

result = subprocess.run(test_cmd, cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('exit code:', result.returncode)

## Implement Batch LIO

In this section, you will utilize your IMU preintegration solution and add the IMU-related constraints needed for a batch factor-graph backend for our LiDAR-inertial SLAM pipeline.

In [src/lio_batch.py](src/lio_batch.py), implement:
- `initialize_backend()`
    - Initializes the factor graph with priors and values.
- `process_lidar_keyframe()`
    - Updates the factor graph and state estimation upon receipt of the next LiDAR keyframe.
- `optimize_batch_graph()`
    - Optimizes over the factor graph with Levenberg-Marquardt.

After implementation, run the cells below. Batch optimization should take about ~5-8 minutes depending on your PC, and should achieve <0.6 ATE RMSE compared to the reference trajectory.

In [ ]:
# Run Batch LIO
from src.lio_batch import BatchLidarImuSlam

slam_batch, result_batch = run_backend(BatchLidarImuSlam, deepcopy(cfg), 'Batch')

In [ ]:
# Evaluate against reference trajectory
fig, ax = plt.subplots(figsize=(7, 7))
gt = None

if slam_batch is not None:
    eval_batch = slam_batch.evaluate_trajectory()
    if eval_batch:
        est = eval_batch['estimated_positions']
        gt = eval_batch['ground_truth_positions']
        ax.plot(est[:, 0], est[:, 1], '-x', label='Batch')
        print('Batch ATE RMSE:', eval_batch.get('ate_rmse'))

if gt is not None:
    ax.plot(gt[:, 0], gt[:, 1], '--', label='Reference trajectory')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Trajectory comparison')
ax.legend()
plt.show()

## Implement ISAM2 LIO

In this section, you will use the same front-end logic as Batch LIO but with an incremental ISAM2 backend for more real-time operation.

In [lio_isam2.py](src/lio_isam2.py), implement:
- `initialize_backend()`
    - Initializes the *pending* factor graph with priors and values.
- `process_lidar_keyframe()`
    - Implement the core ISAM2 update logic upon receipt of the next LiDAR keyframe.
- `update_isam2()`
    - Update ISAM2 with the pending factor graph and values.

After implementation, run the cells below. ISAM2 optimization should take about ~2-4 minutes depending on your PC, and should achieve <0.4 ATE RMSE compared to the reference trajectory.

In [ ]:
# Run ISAM2 LIO
from src.lio_isam2 import Isam2LidarImuSlam

slam_isam, result_isam = run_backend(Isam2LidarImuSlam, deepcopy(cfg), 'ISAM2')

In [ ]:
# Evaluate against reference trajectory
fig, ax = plt.subplots(figsize=(7, 7))
gt = None

if slam_isam is not None:
    eval_isam = slam_isam.evaluate_trajectory()
    if eval_isam:
        est = eval_isam['estimated_positions']
        gt = eval_isam['ground_truth_positions']
        ax.plot(est[:, 0], est[:, 1], '-o', label='ISAM2')
        print('ISAM2 ATE RMSE:', eval_isam.get('ate_rmse'))

if gt is not None:
    ax.plot(gt[:, 0], gt[:, 1], '--', label='Reference trajectory')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Trajectory comparison')
ax.legend()
plt.show()

## Compare Optimization Update Times

Run the cell below to visualize the optimization update times for each of the backends. Observe how each method's update times change as the number of keyframes (variables) in the graph grows.

If you correctly optimized over your backends and recorded their update times, you should see a quadratically growing optimization time as the number of keyframes grows for batch LIO (O(n^2)) while ISAM2 update times stay relatively constant (O(1)).

In [ ]:
# Plot backend update times
batch_times = getattr(result_batch, 'update_times_sec', []) if result_batch is not None else []
isam_times = getattr(result_isam, 'update_times_sec', []) if result_isam is not None else []

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if batch_times:
    axes[0].plot(np.arange(len(batch_times)), batch_times, '-x', label='Batch')
if isam_times:
    axes[0].plot(np.arange(len(isam_times)), isam_times, '-o', label='ISAM2')
axes[0].set_xlabel('update index')
axes[0].set_ylabel('time (s)')
axes[0].set_title('Per-update time')
axes[0].legend()

if batch_times:
    axes[1].plot(np.arange(len(batch_times)), np.cumsum(batch_times), '-x', label='Batch cumulative')
if isam_times:
    axes[1].plot(np.arange(len(isam_times)), np.cumsum(isam_times), '-o', label='ISAM2 cumulative')
axes[1].set_xlabel('update index')
axes[1].set_ylabel('cumulative time (s)')
axes[1].set_title('Cumulative time')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Batch: count=', len(batch_times), 'mean(s)=', mean(batch_times), 'median(s)=', median(batch_times))
print('ISAM2: count=', len(isam_times), 'mean(s)=', mean(isam_times), 'median(s)=', median(isam_times))

## Compare IMU Preintegration Usage

Run the cells below to observe the performance impact (relative to our reference trajectory) of using IMU preintegration as a source of constraints in our batch optimization.

You should observe a slight degradation in ATE RMSE relative to the reference trajectory when IMU preintegration is used. 

The reflection questions will encourage you to consider why this might be the case for our specific context. 
* One potentially non-obvious factor is that we are starting our odometry mid-trajectory relative to the ROS bag. How and why might this impact the usefulness of IMU preintegration?

In [ ]:
slice_len = 2000

cfg_noimu = deepcopy(cfg)
cfg_noimu.use_preintegration = False
cfg_noimu.stream_end_index = cfg_noimu.stream_start_index + slice_len
cfg_noimu.voxel_size_m = 0.10  # finer voxels for high-res comparison

cfg_imu = deepcopy(cfg)
cfg_imu.use_preintegration = True
cfg_imu.stream_end_index = cfg_noimu.stream_end_index
cfg_imu.voxel_size_m = cfg_noimu.voxel_size_m

print('Running Batch without IMU on the standardized slice...')
slam_noimu, result_noimu = run_backend(BatchLidarImuSlam, cfg_noimu, 'Batch without IMU')

print('Running Batch with IMU on the standardized slice...')
slam_imu, result_imu = run_backend(BatchLidarImuSlam, cfg_imu, 'Batch with IMU')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
gt = None

eval_noimu = slam_noimu.evaluate_trajectory()
if eval_noimu:
    est = eval_noimu['estimated_positions']
    gt = eval_noimu['ground_truth_positions']
    ax.plot(est[:, 0], est[:, 1], '-o', label='Batch no IMU')
    print('No-IMU ATE RMSE:', eval_noimu.get('ate_rmse'))

eval_imu = slam_imu.evaluate_trajectory()
if eval_imu:
    est = eval_imu['estimated_positions']
    gt = eval_imu['ground_truth_positions']
    ax.plot(est[:, 0], est[:, 1], '-x', label='Batch with IMU')
    print('With-IMU ATE RMSE:', eval_imu.get('ate_rmse'))

if gt is not None:
    ax.plot(gt[:, 0], gt[:, 1], '--', label='Reference trajectory')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('With vs without IMU preintegration')
ax.legend()
plt.show()

## Visualize Point-Cloud Map

Use Open3D to view your stitched point clouds!

**TIP**: The Open3D window that pops up can be closed by pressing `Q`. After closing the window, the application that opened it may still be running. Don't force close it, as it will cause your Jupyter Notebook kernel to restart, erasing your results. Instead, just refocus VSCode and you can run other cells. After you are finished, manually restart your kernel to close the application.

In [ ]:
# Build and visualize the global map using Open3D
import open3d as o3d

def visualize_map(points, downsample_voxel=0.0):
    if points is None or len(points) == 0:
        print('No points to visualize.')
        return
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(points)
    if downsample_voxel and downsample_voxel > 0:
        pc = pc.voxel_down_sample(downsample_voxel)

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name='Global Map')
    vis.add_geometry(pc)
    
    # Blocks here until you press 'Q' or close the window
    vis.run() 
    
    # 4. Explicitly kill the window to free the Jupyter kernel
    vis.destroy_window()

In [ ]:
# Visualize slam_batch
points = slam_batch.build_global_map()

print('Points shape:', points.shape)
visualize_map(points, downsample_voxel=0.05)

In [ ]:
# Visualize slam_isam
points = slam_isam.build_global_map()

print('Points shape:', points.shape)
visualize_map(points, downsample_voxel=0.05)